<a href="https://colab.research.google.com/github/Chosencodes/Left-Atrium-Segmentation-from-3D-Cardiac-MRI/blob/main/Atrium_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from tqdm.notebook import tqdm
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
root = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/imagesTr")
label = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/labelsTr")

In [ ]:
sample_path = next(root.glob("la*.nii.gz"))

In [ ]:
sample_path_label = label / sample_path.name

In [ ]:
sample_path, sample_path_label

In [ ]:
data = nib.load(sample_path)
label = nib.load(sample_path_label)

mri = data.get_fdata()
mask = label.get_fdata().astype(np.uint8)

In [ ]:
mri.shape,mask.shape

In [ ]:
nib.aff2axcodes(data.affine)

In [ ]:
slice_num = 40

plt.figure(figsize=(6,6))
plt.imshow(mri[:,:,slice_num], cmap="bone")

mask_ = np.ma.masked_where(mask[:,:,slice_num] == 0,
                           mask[:,:,slice_num])

plt.imshow(mask_, alpha=0.5, cmap="autumn")

plt.show()

# **Preprocessing and Creating the Dataset**

In [ ]:
!pip install torchio nibabel scikit-learn tqdm -q

In [ ]:
import torchio as tio
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

In [ ]:
root = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/imagesTr")
label = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/labelsTr")

In [ ]:
image_path = sorted(root.glob("*.nii.gz"))
label_path = sorted(label.glob("*.nii.gz"))

In [ ]:
def make_subjects(image_path, label_path):
  subjects = []

  for img_path,lbl_path in zip(image_path, label_path):
    subject = tio.Subject(
        mri = tio.ScalarImage(img_path),
        label = tio.LabelMap(lbl_path)
    )
    subjects.append(subject)
  return subjects

all_subjects = make_subjects(image_path, label_path)

In [ ]:
# subject = all_subjects[0]
# mri = subject["mri"]
# print(f"Raw shape: {mri.shape}")
# print(f"Depth (D): {mri.shape[-1]}")

In [ ]:
train_subject,val_subject=train_test_split(all_subjects,test_size=0.15,random_state=42)

In [ ]:
base_transform = tio.Compose([
    tio.ToCanonical(),
    tio.CropOrPad((208, 208, 128), mask_name="label"),
    tio.ZNormalization(masking_method=tio.ZNormalization.mean),
    tio.Clamp(out_min=-3, out_max=3),
    tio.RescaleIntensity(percentiles=(0, 100)),
])

In [ ]:
train_transform = tio.Compose([
    base_transform,
    tio.RandomFlip(axes=(0, 1, 2), flip_probability=0.5),
    tio.RandomAffine(scales=0.05, degrees=5, translation=3),
    tio.RandomNoise(mean=0, std=0.01),
])

val_transform=tio.Compose([
    base_transform
])

In [ ]:
train_dataset = tio.SubjectsDataset(train_subject, transform=train_transform)
val_dataset   = tio.SubjectsDataset(val_subject,   transform=val_transform)

In [ ]:
sample = train_dataset[0]
mri_tensor   = sample["mri"]["data"]
label_tensor = sample["label"]["data"]

print(f"   MRI   : {mri_tensor.shape}")
print(f"   Label : {label_tensor.shape}")
print(f" MRI value range : [{mri_tensor.min():.3f}, {mri_tensor.max():.3f}]")
print(f"Label unique vals: {label_tensor.unique()}")

In [ ]:
# for i in range(16):
#     sample = train_dataset[i]
#     img = sample["mri"]["data"]
#     print(i, img.min().item(), img.max().item())

In [ ]:
def show_slice(subject, slice_idx=50):
    mri   = subject["mri"]["data"][0]
    label = subject["label"]["data"][0]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    views = {
        "Axial (top-down)"  : (mri[:,:,slice_idx],   label[:,:,slice_idx]),
        "Coronal (front)"   : (mri[:,slice_idx,:],   label[:,slice_idx,:]),
        "Sagittal (side)"   : (mri[slice_idx,:,:],   label[slice_idx,:,:]),
    }
    for ax, (title, (img, msk)) in zip(axes, views.items()):
        ax.imshow(img, cmap="bone")
        masked = np.ma.masked_where(msk == 0, msk)
        ax.imshow(masked, alpha=0.5, cmap="autumn")
        ax.set_title(title)
        ax.axis("off")

    plt.suptitle("MRI Scan + Atrium Mask (red overlay)", fontsize=14)
    plt.tight_layout()
    plt.show()

show_slice(val_dataset[0], slice_idx=50)

In [ ]:

fig, axes = plt.subplots(3, 3, figsize=(9, 9))

for i in range(3):
    for j in range(3):
        sample = train_dataset[3]
        mri    = sample["mri"]["data"]
        mask   = sample["label"]["data"]

        d = mri.shape[-1] // 2

        slice_2d = mri[0, :, :, d]
        mask_2d  = mask[0, :, :, d]

        mask_ = np.ma.masked_where(mask_2d == 0, mask_2d)

        axes[i][j].imshow(slice_2d, cmap="bone")
        axes[i][j].imshow(mask_, alpha=0.5, cmap="autumn")
        axes[i][j].axis("off")

fig.suptitle("Sample augmentations — same subject, 9 different augmentations", fontsize=12)
plt.tight_layout()
plt.show()

# **Model**

In [ ]:
!pip install monai pytorch-lightning -q

In [ ]:

from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

In [ ]:
class AtriumDataset(Dataset):
    def __init__(self, subjects_dataset):
        self.dataset = subjects_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        mri    = sample["mri"]["data"]
        label  = sample["label"]["data"]
        return mri, label

In [ ]:
train_loader  = DataLoader(AtriumDataset(train_dataset), batch_size=1, num_workers=0, shuffle=True)
val_loader    = DataLoader(AtriumDataset(val_dataset),   batch_size=1, num_workers=0, shuffle=False)

In [ ]:
def build_model():
    return UNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=1,
        channels=(32,64,128,256,512),
        strides=(2,2,2,2),
        num_res_units=2,
        dropout=0.1,
    )

In [ ]:
class AtriumSegmentation(pl.LightningModule):
    def __init__(self,lr=1e-4):
        super().__init__()
        self.save_hyperparameters()

        self.model = build_model()
        self.loss = DiceCELoss(sigmoid=True,lambda_dice=0.7,lambda_ce=0.3)

        self.metrics = DiceMetric(include_background=True,reduction="mean")

    def forward(self,x):
        return self.model(x)

    def training_step(self,batch,batch_idx):
        mri, label = batch
        label = label.float()
        pred = self.model(mri)
        loss = self.loss(pred,label)

        self.log("train dice loss",loss,prog_bar=True)

        if batch_idx % 50 ==0:
            self.log_images(mri.cpu(),pred.cpu(),label.cpu(),"train")

        return loss

    def validation_step(self,batch,batch_idx):
        mri, label = batch
        label = label.float()
        pred = self.model(mri)
        loss = self.loss(pred,label)

        pred_binary = (torch.sigmoid(pred) > 0.5).float()
        self.metrics(pred_binary, label)

        self.log("val dice loss",loss,prog_bar=True)

        if batch_idx % 2 == 0:
            self.log_images(mri.cpu(),pred.cpu(),label.cpu(),"val")

        return loss

    def on_validation_epoch_end(self):
        dice_score = self.metrics.aggregate().item()
        self.metrics.reset()
        self.log("val dice score",dice_score,prog_bar=True)
        print(f"Val dice score: {dice_score:.4f}")

    def log_images(self,mri,pred,mask,name):
        d = mri.shape[-1] // 2
        pred_binary = (torch.sigmoid(pred) > 0.5).float()

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))

        axes[0].imshow(mri[0, 0, :, :, d],  cmap="bone")
        gt = np.ma.masked_where(mask[0, 0, :, :, d] == 0,
                                mask[0, 0, :, :, d])
        axes[0].imshow(gt, alpha=0.6, cmap="autumn")
        axes[0].set_title("Ground Truth")
        axes[0].axis("off")

        axes[1].imshow(mri[0, 0, :, :, d],  cmap="bone")
        pr = np.ma.masked_where(pred_binary[0, 0, :, :, d] == 0,
                                pred_binary[0, 0, :, :, d])
        axes[1].imshow(pr, alpha=0.6, cmap="autumn")
        axes[1].set_title("Prediction")
        axes[1].axis("off")

        plt.suptitle(f"{name} — depth slice {d}", fontsize=12)
        plt.tight_layout()
        self.logger.experiment.add_figure(name, fig, self.global_step)
        plt.close(fig)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val dice loss"},
        }

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor    = "val dice score",
    mode       = "max",
    save_top_k = 3,
    filename   = "best-model",
)

early_stop_callback = EarlyStopping(
    monitor  = "val dice score",
    mode     = "max",
    patience =  15,
)

In [ ]:
torch.manual_seed(42)
model = AtriumSegmentation()

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
trainer = pl.Trainer(
    accelerator                 = "gpu" if torch.cuda.is_available() else "cpu",
    devices                     = 1,
    max_epochs                  = 150,
    gradient_clip_val           = 0.5,
    accumulate_grad_batches     = 4,
    logger                      = TensorBoardLogger(save_dir="./logs", name="atrium"),
    callbacks                   = [checkpoint_callback, early_stop_callback],
    log_every_n_steps           = 1,
)

In [ ]:
trainer.fit(model, train_loader, val_loader)

# **Evalution**

In [ ]:
import shutil
from monai.metrics import DiceMetric, HausdorffDistanceMetric

In [ ]:
src = checkpoint_callback.best_model_path
dst = "/content/drive/MyDrive/Atrium-Segmentation/best-model.ckpt"
shutil.copy(src, dst)
print("Saved to:", dst)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AtriumSegmentation.load_from_checkpoint(
    "/content/drive/MyDrive/Atrium-Segmentation/best-model.ckpt"
)
model.eval()
model.to(device)

In [ ]:
dice_metric = DiceMetric(include_background=False, reduction="mean")
hausdorff_metric = HausdorffDistanceMetric(include_background=False, reduction="mean")

In [ ]:
from monai.transforms import KeepLargestConnectedComponent

keep_largest = KeepLargestConnectedComponent(applied_labels=[1])

preds_all  = []
labels_all = []

for mri, label in tqdm(val_loader, desc="Evaluating"):
    mri   = mri.to(device)
    label = label.float().to(device)

    with torch.no_grad():
        pred        = model(mri)
        pred_binary = (torch.sigmoid(pred) > 0.5).float()
        pred_binary = keep_largest(pred_binary.cpu()).to(device)

    dice_metric(pred_binary, label)
    hausdorff_metric(pred_binary.cpu(), label.cpu())

    preds_all.append(pred_binary.cpu())
    labels_all.append(label.cpu())

dice_score      = dice_metric.aggregate().item()
hausdorff_score = hausdorff_metric.aggregate().item()
dice_metric.reset()
hausdorff_metric.reset()

print(f"Dice score: {dice_score:.4f}")
print(f"Hausdorff distance: {hausdorff_score:.4f}")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for idx, (mri, label) in enumerate(val_loader):
    mri   = mri.to(device)
    label = label.float()

    with torch.no_grad():
        pred        = model(mri)
        pred_binary = (torch.sigmoid(pred) > 0.5).float().cpu()

    mri_np = mri.cpu()[0, 0]
    depths = [20, mri_np.shape[-1]//2, mri_np.shape[-1]-20]

    for col, d in enumerate(depths):
        ax = axes[idx][col]
        ax.imshow(mri_np[:, :, d], cmap="bone")


        gt = np.ma.masked_where(label[0,0,:,:,d]==0, label[0,0,:,:,d])
        ax.imshow(gt, alpha=0.5, cmap="summer")


        pr = np.ma.masked_where(pred_binary[0,0,:,:,d]==0,
                                pred_binary[0,0,:,:,d])
        ax.imshow(pr, alpha=0.5, cmap="autumn")

        ax.set_title(f"Subject {idx+1} | slice {d}")
        ax.axis("off")

plt.suptitle("Green=Ground Truth  Red=Prediction  Orange=Overlap", fontsize=12)
plt.tight_layout()
plt.show()